# Model: patronus-studio/wolf-defender-prompt-injection

**main-v1**: the best measures from the three competitor notebooks, in one recipe.

| Measure | From |
|---|---|
| Backbone `wolf-defender-prompt-injection` (safety-tuned ModernBERT). Its 2-class head is replaced by a new 3-class one | 0.97 team |
| lr 3e-5, effective batch 64, 4 epochs, keep the best epoch | 0.97 team |
| max_len 384; long prompts keep their first 2/3 and last 1/3 | 0.94 team |
| Retrain on 100% of train for the best epoch count | 0.94 team |
| Length-grouped batches; class offsets tuned on the holdout, used only if they gain > 0.003 | 0.93 team |
| 10% holdout stratified by label + language, 1/√freq class weights, fp16 | all three |

Left out: the 0.94 team's mean-pooling head (wolf-defender's own pretrained head is kept) and their auxiliary labels (they scored lower on their own validation).

**Kaggle settings:** GPU T4 (one is used), Internet off. Attach the competition data and the `wolf-defender-prompt-injection` model. About 30–40 min.

Writes `/kaggle/working/submission.csv` (retrained model) and `submission_holdout_model.csv` (backup from the 90% model).

In [ ]:
import glob, math, os, time
import numpy as np, pandas as pd, torch, transformers
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

MODEL_NAME = "wolf-defender"
MAX_LEN, HEAD_FRAC = 384, 2 / 3
BS, ACCUM, EPOCHS, LR, WARMUP, WD = 16, 4, 4, 3e-5, 0.1, 0.01  # effective batch = BS * ACCUM = 64
VAL_FRAC, SEED, REFIT_FULL = 0.10, 42, True
LABELS = ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]

transformers.logging.set_verbosity_error()  # hides the harmless "LOAD REPORT" about the new 3-class head
rng = np.random.default_rng(SEED)
dev = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

DATA_DIR = os.path.dirname(glob.glob("/kaggle/input/**/train.csv", recursive=True)[0])
models = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/config.json", recursive=True) if MODEL_NAME in p.lower()]
assert models, f"attach the {MODEL_NAME} model as an input"
print("data:", DATA_DIR, "| model:", models[0])

In [ ]:
train, test = pd.read_csv(f"{DATA_DIR}/train.csv"), pd.read_csv(f"{DATA_DIR}/test.csv")
y = train.label.map({l: i for i, l in enumerate(LABELS)}).values
tr, va = train_test_split(np.arange(len(train)), test_size=VAL_FRAC, random_state=SEED,
                          stratify=train.label + "_" + train.language)
print("train / holdout rows:", len(tr), len(va))

In [ ]:
tok = AutoTokenizer.from_pretrained(models[0], local_files_only=True)
core, full = tok("hello world", add_special_tokens=False)["input_ids"], tok("hello world")["input_ids"]
cut = next(i for i in range(len(full)) if full[i:i + len(core)] == core)
PRE, POST = full[:cut], full[cut + len(core):]  # this tokenizer's own special tokens, e.g. [CLS] ... [SEP]
BUDGET = MAX_LEN - len(PRE) - len(POST)
HEAD = int(BUDGET * HEAD_FRAC)

def encode(texts):
    # long prompts keep their first HEAD and last BUDGET - HEAD tokens
    return [PRE + (x if len(x) <= BUDGET else x[:HEAD] + x[len(x) - BUDGET + HEAD:]) + POST
            for x in tok(list(texts), add_special_tokens=False)["input_ids"]]

X, X_te = encode(train.text), encode(test.text)
assert max(map(len, X + X_te)) <= MAX_LEN and encode(["hello world"])[0] == full
print(f"prompts cut at {MAX_LEN} tokens: {np.mean([len(x) == MAX_LEN for x in X]):.1%}")

def pad(seqs):
    ids = torch.full((len(seqs), max(map(len, seqs))), tok.pad_token_id)
    mask = torch.zeros_like(ids)
    for r, s in enumerate(seqs):
        ids[r, :len(s)], mask[r, :len(s)] = torch.tensor(s), 1
    return {"input_ids": ids.to(dev), "attention_mask": mask.to(dev)}

def train_batches(idx):
    # sort by length inside chunks of 50 batches: little padding, order still random
    perm, out = rng.permutation(idx), []
    for i in range(0, len(perm), BS * 50):
        c = perm[i:i + BS * 50]
        c = c[np.argsort([len(X[j]) for j in c])]
        out += [c[k:k + BS] for k in range(0, len(c), BS)]
    return [out[i] for i in rng.permutation(len(out))]

@torch.no_grad()
def predict(model, seqs, bs=64):
    model.eval()
    order, out = np.argsort([len(s) for s in seqs]), np.zeros((len(seqs), 3), dtype=np.float32)
    for i in range(0, len(seqs), bs):
        b = order[i:i + bs]
        with torch.autocast("cuda", dtype=torch.float16):
            out[b] = model(**pad([seqs[j] for j in b])).logits.float().log_softmax(-1).cpu().numpy()
    return out

In [ ]:
def fit(train_idx, epochs, val_idx=None):
    # returns (holdout macro-F1, epoch, holdout log-probs, test log-probs) for the best epoch, or the last one without a holdout
    torch.manual_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(  # its 2-class head is replaced by a new 3-class one
        models[0], num_labels=3, ignore_mismatched_sizes=True, local_files_only=True).to(dev)
    w = 1 / np.sqrt(np.bincount(y[train_idx], minlength=3) / len(train_idx))
    loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor(w / w.mean(), dtype=torch.float32, device=dev))
    no_decay = lambda n: "bias" in n or "norm" in n.lower()
    opt = torch.optim.AdamW([
        {"params": [p for n, p in model.named_parameters() if not no_decay(n)], "weight_decay": WD},
        {"params": [p for n, p in model.named_parameters() if no_decay(n)], "weight_decay": 0.0}], lr=LR)
    steps = epochs * math.ceil(math.ceil(len(train_idx) / BS) / ACCUM)  # optimizer steps
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP * steps), steps)
    scaler = torch.amp.GradScaler("cuda")
    best = None
    for ep in range(1, epochs + 1):
        model.train()
        t0, batches = time.time(), train_batches(train_idx)
        for i, b in enumerate(batches, 1):
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(**pad([X[j] for j in b])).logits
            loss = loss_fn(logits.float(), torch.tensor(y[b], device=dev))
            scaler.scale(loss / ACCUM).backward()
            if i % ACCUM == 0 or i == len(batches):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
            if i == 50 or i % 400 == 0:
                print(f"ep{ep} step {i}/{len(batches)} loss {loss.item():.4f} | "
                      f"{(time.time() - t0) / i * (len(batches) - i) / 60:.1f} min left in epoch", flush=True)
        if val_idx is None:
            print(f"== epoch {ep} done ({(time.time() - t0) / 60:.1f} min)")
            continue
        lv = predict(model, [X[j] for j in val_idx])
        f1 = f1_score(y[val_idx], lv.argmax(1), average="macro")
        print(f"== epoch {ep}: holdout macro-F1 {f1:.4f} ({(time.time() - t0) / 60:.1f} min)")
        if best is None or f1 > best[0]:  # keep the best epoch's predictions, so no weights need saving
            best = (f1, ep, lv, predict(model, X_te))
    return best or (None, epochs, None, predict(model, X_te))

f1_va, best_ep, lv, lt_holdout = fit(tr, EPOCHS, va)

In [ ]:
yv = y[va]
print(f"best epoch {best_ep}/{EPOCHS}, holdout macro-F1 {f1_va:.4f}")
print(classification_report(yv, lv.argmax(1), target_names=LABELS, digits=4))
for lang, m in train.iloc[va].groupby("language").indices.items():
    print(f"{lang}: macro-F1 {f1_score(yv[m], lv[m].argmax(1), average='macro'):.4f} (n={len(m)})")

# class offsets added to the log-probs, on a coarse grid; kept only if the gain isn't noise on ~1,400 rows
raw = f1_score(yv, lv.argmax(1), average="macro")
best = (raw, 0.0, 0.0)
for bj in np.arange(-1.5, 1.51, 0.25):
    for bp in np.arange(-1.5, 1.51, 0.25):
        f = f1_score(yv, (lv + [0, bj, bp]).argmax(1), average="macro")
        if f > best[0] + 1e-9:
            best = (f, bj, bp)
OFFSET = np.array([0, best[1], best[2]]) if best[0] - raw > 0.003 else np.zeros(3)
print(f"holdout macro-F1 raw {raw:.4f} -> tuned {best[0]:.4f} | offsets used: JB {OFFSET[1]:+.2f}, PI {OFFSET[2]:+.2f}")

In [ ]:
def write(lt, path):
    pred = (lt + OFFSET).argmax(1)
    sub = pd.DataFrame({"id": test.id, "label": np.array(LABELS)[pred]})
    assert len(sub) == 6000 and sub.id.is_unique and set(sub.id) == set(test.id) and sub.label.isin(LABELS).all()
    sub.to_csv(path, index=False)
    print(path, sub.label.value_counts(normalize=True).round(3).to_dict(), "| train: 0.62 / 0.31 / 0.07")

write(lt_holdout, "/kaggle/working/submission_holdout_model.csv")  # backup in case the retrain is interrupted
# retrain on all 14,000 rows for the best epoch count; the holdout offsets carry over
lt = fit(np.arange(len(train)), best_ep)[3] if REFIT_FULL else lt_holdout
write(lt, "/kaggle/working/submission.csv")